# MNIST with a Convolutional Neural Network (CNN)

## Building on Our Simple Network

---

In the previous notebook, we built a simple neural network that achieved ~97-98% accuracy on MNIST. Now we'll use a **Convolutional Neural Network (CNN)** to do even better!

**What you will learn:**
1. Why CNNs are better for images
2. How convolution works
3. How to build a CNN in PyTorch
4. How to compare CNN vs Simple NN

---

## Why CNNs for Images?

**Problems with simple neural networks for images:**

1. **Too many parameters**: A 28×28 image has 784 pixels. If our first layer has 128 neurons, that's already 100,000 parameters!

2. **Ignores spatial structure**: When we flatten an image, we lose the information that neighboring pixels are related.

3. **No translation invariance**: A "3" in the top-left looks completely different (as raw pixels) from a "3" in the center.

**CNNs solve these problems by:**
- Using small filters that slide across the image
- Sharing weights across the image (parameter efficiency)
- Detecting features regardless of position

---

## Step 1: Setup (Same as Before)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Device
# Device konfigurieren
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Device: {device}')

## Step 2: Load Data (Same as Before)

In [ ]:
# Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Datasets
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

# Data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training: {len(train_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

## Step 3: Understanding Convolution

**What is a convolution?**

A convolution slides a small filter (e.g., 3×3) across the image:

```
Image (5x5)          Filter (3x3)        Output
┌─────────────┐      ┌───────┐           
│ 1 2 3 4 5   │      │ 1 0 1 │           
│ 6 7 8 9 10  │  *   │ 0 1 0 │   =   Feature Map
│ 11...       │      │ 1 0 1 │           
└─────────────┘      └───────┘           
```

At each position:
1. Multiply filter values with image values
2. Sum them up
3. Write result to output

**The filter learns to detect patterns!**
- Edge detectors
- Corner detectors
- Texture patterns

Let's visualize this:

In [ ]:
# Get a sample image
sample_image, sample_label = train_dataset[0]
print(f"Image shape: {sample_image.shape}")
print(f"Label: {sample_label}")

# Create some example filters
# Horizontal edge detector
horizontal_filter = torch.tensor([[[-1, -1, -1],
                                   [ 0,  0,  0],
                                   [ 1,  1,  1]]], dtype=torch.float32)

# Vertical edge detector
vertical_filter = torch.tensor([[[-1, 0, 1],
                                 [-1, 0, 1],
                                 [-1, 0, 1]]], dtype=torch.float32)

# Apply convolutions manually
import torch.nn.functional as F

# Add batch dimension
img = sample_image.unsqueeze(0)  # Shape: (1, 1, 28, 28)

# Apply filters
horizontal_edges = F.conv2d(img, horizontal_filter.unsqueeze(0), padding=1)
vertical_edges = F.conv2d(img, vertical_filter.unsqueeze(0), padding=1)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(sample_image.squeeze() * 0.5 + 0.5, cmap='gray')
axes[0].set_title(f'Original (Label: {sample_label})')
axes[0].axis('off')

axes[1].imshow(horizontal_edges.squeeze().detach(), cmap='gray')
axes[1].set_title('Horizontal Edges')
axes[1].axis('off')

axes[2].imshow(vertical_edges.squeeze().detach(), cmap='gray')
axes[2].set_title('Vertical Edges')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\nNotice how the filters highlight different features!")

## Step 4: Understanding Pooling

**What is pooling?**

Pooling reduces the size of feature maps:

```
Max Pooling (2×2):

┌─────┬─────┐        ┌─────┐
│ 1 3 │ 2 4 │        │ 4 6 │
│ 5 2 │ 6 1 │   →    │ 8 7 │
├─────┼─────┤        └─────┘
│ 3 8 │ 2 7 │
│ 1 4 │ 5 3 │
└─────┴─────┘
```

**Benefits:**
- Reduces computation
- Provides some translation invariance
- Keeps the most important features (max values)

In [ ]:
# Demonstrate pooling
pool = nn.MaxPool2d(kernel_size=2, stride=2)

# Create example
example = torch.tensor([[[[1, 3, 2, 4],
                          [5, 2, 6, 1],
                          [3, 8, 2, 7],
                          [1, 4, 5, 3]]]], dtype=torch.float32)

pooled = pool(example)

print("Before pooling:")
print(example.squeeze().numpy())
print(f"\nShape: {example.shape}")

print("\nAfter max pooling (2×2):")
print(pooled.squeeze().numpy())
print(f"\nShape: {pooled.shape}")

## Step 5: Build the CNN

Our CNN architecture:

```
Input (1, 28, 28)
    ↓
Conv2d (32 filters, 3×3) → ReLU → MaxPool (2×2)
    ↓
Conv2d (64 filters, 3×3) → ReLU → MaxPool (2×2)
    ↓
Flatten
    ↓
Linear (128) → ReLU
    ↓
Linear (10)
    ↓
Output (10 classes)
```

**Dimension tracking:**
- Input: 1 × 28 × 28
- After Conv1: 32 × 28 × 28 (same padding)
- After Pool1: 32 × 14 × 14
- After Conv2: 64 × 14 × 14
- After Pool2: 64 × 7 × 7 = 3136
- After Flatten: 3136
- After FC1: 128
- Output: 10

In [ ]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for MNIST digit classification.
    """
    
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Convolutional layers
        # Conv2d(in_channels, out_channels, kernel_size, padding)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        # Pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        # After 2 pooling layers: 28 -> 14 -> 7
        # So we have 64 channels × 7 × 7 = 3136
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        
        # Activation
        self.relu = nn.ReLU()
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input tensor (batch_size, 1, 28, 28)
        
        Returns:
            Output tensor (batch_size, 10)
        """
        # First conv block: Conv -> ReLU -> Pool
        x = self.conv1(x)        # (batch, 32, 28, 28)
        x = self.relu(x)
        x = self.pool(x)         # (batch, 32, 14, 14)
        
        # Second conv block: Conv -> ReLU -> Pool
        x = self.conv2(x)        # (batch, 64, 14, 14)
        x = self.relu(x)
        x = self.pool(x)         # (batch, 64, 7, 7)
        
        # Flatten for fully connected layers
        x = x.view(-1, 64 * 7 * 7)  # (batch, 3136)
        
        # Fully connected layers
        x = self.fc1(x)          # (batch, 128)
        x = self.relu(x)
        x = self.fc2(x)          # (batch, 10)
        
        return x

In [ ]:
# Create model
cnn_model = SimpleCNN().to(device)
print(cnn_model)

# Count parameters
cnn_params = sum(p.numel() for p in cnn_model.parameters())
print(f"\nTotal parameters: {cnn_params:,}")

### Let's Verify the Dimensions

Let's trace a sample through the network to verify our calculations:

In [ ]:
# Test with a sample batch
sample_batch = torch.randn(1, 1, 28, 28).to(device)

print("Tracing dimensions through the CNN:")
print(f"Input:        {sample_batch.shape}")

# Manually trace
x = cnn_model.conv1(sample_batch)
print(f"After Conv1:  {x.shape}")
x = cnn_model.relu(x)
x = cnn_model.pool(x)
print(f"After Pool1:  {x.shape}")

x = cnn_model.conv2(x)
print(f"After Conv2:  {x.shape}")
x = cnn_model.relu(x)
x = cnn_model.pool(x)
print(f"After Pool2:  {x.shape}")

x = x.view(-1, 64 * 7 * 7)
print(f"After Flatten: {x.shape}")

x = cnn_model.fc1(x)
print(f"After FC1:    {x.shape}")

x = cnn_model.fc2(x)
print(f"Output:       {x.shape}")

## Step 6: Training Functions

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track stats
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(train_loader), 100 * correct / total


def evaluate(model, test_loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(test_loader), 100 * correct / total

## Step 7: Train the CNN

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Training
num_epochs = 10
cnn_history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("Training CNN...")
print("-" * 60)

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        cnn_model, train_loader, criterion, optimizer, device
    )
    test_loss, test_acc = evaluate(
        cnn_model, test_loader, criterion, device
    )
    
    cnn_history['train_loss'].append(train_loss)
    cnn_history['train_acc'].append(train_acc)
    cnn_history['test_loss'].append(test_loss)
    cnn_history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

print("-" * 60)
print(f"Final CNN test accuracy: {test_acc:.2f}%")

## Step 8: Compare CNN vs Simple NN

Let's train the simple NN again for comparison:

In [ ]:
# Simple NN from previous notebook
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

# Train simple NN
simple_model = SimpleNN().to(device)
simple_optimizer = optim.Adam(simple_model.parameters(), lr=0.001)
simple_history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("Training Simple NN for comparison...")
print("-" * 60)

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        simple_model, train_loader, criterion, simple_optimizer, device
    )
    test_loss, test_acc = evaluate(
        simple_model, test_loader, criterion, device
    )
    
    simple_history['train_loss'].append(train_loss)
    simple_history['train_acc'].append(train_acc)
    simple_history['test_loss'].append(test_loss)
    simple_history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

print("-" * 60)
print(f"Final Simple NN test accuracy: {test_acc:.2f}%")

In [ ]:
# Compare the two models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, num_epochs + 1)

# Loss comparison
axes[0].plot(epochs, simple_history['test_loss'], 'b-', label='Simple NN', linewidth=2)
axes[0].plot(epochs, cnn_history['test_loss'], 'r-', label='CNN', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Loss')
axes[0].set_title('Test Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
axes[1].plot(epochs, simple_history['test_acc'], 'b-', label='Simple NN', linewidth=2)
axes[1].plot(epochs, cnn_history['test_acc'], 'r-', label='CNN', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
simple_params = sum(p.numel() for p in simple_model.parameters())

print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(f"{'Metric':<25} {'Simple NN':<15} {'CNN':<15}")
print("-"*50)
print(f"{'Parameters':<25} {simple_params:,}" + " "*6 + f"{cnn_params:,}")
print(f"{'Final Test Accuracy':<25} {simple_history['test_acc'][-1]:.2f}%" + " "*8 + f"{cnn_history['test_acc'][-1]:.2f}%")
print(f"{'Final Test Loss':<25} {simple_history['test_loss'][-1]:.4f}" + " "*9 + f"{cnn_history['test_loss'][-1]:.4f}")
print("="*50)

## Step 9: Visualize What the CNN Learns

Let's look at the filters the CNN learned:

In [ ]:
# Get the weights from the first convolutional layer
conv1_weights = cnn_model.conv1.weight.data.cpu()
print(f"Conv1 filter shape: {conv1_weights.shape}")
print(f"  - 32 filters, each is 1×3×3")

# Visualize the first 16 filters
fig, axes = plt.subplots(4, 8, figsize=(14, 7))

for i, ax in enumerate(axes.flat):
    if i < 32:
        filter_img = conv1_weights[i, 0]  # Shape: (3, 3)
        ax.imshow(filter_img, cmap='gray')
        ax.set_title(f'Filter {i+1}', fontsize=8)
    ax.axis('off')

plt.suptitle('Learned Filters in Conv1 Layer', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize feature maps for a sample image
sample_image, sample_label = test_dataset[0]
sample_image = sample_image.unsqueeze(0).to(device)

# Get activations after first conv layer
cnn_model.eval()
with torch.no_grad():
    conv1_output = cnn_model.relu(cnn_model.conv1(sample_image))

# Plot original and some feature maps
fig, axes = plt.subplots(3, 6, figsize=(14, 7))

# Original image
axes[0, 0].imshow(sample_image.cpu().squeeze() * 0.5 + 0.5, cmap='gray')
axes[0, 0].set_title(f'Original\n(Label: {sample_label})')
axes[0, 0].axis('off')

# Hide remaining in first row
for i in range(1, 6):
    axes[0, i].axis('off')

# Feature maps
for i in range(12):
    row = (i // 6) + 1
    col = i % 6
    feature_map = conv1_output[0, i].cpu()
    axes[row, col].imshow(feature_map, cmap='viridis')
    axes[row, col].set_title(f'Filter {i+1}', fontsize=8)
    axes[row, col].axis('off')

plt.suptitle('Feature Maps After Conv1 Layer', fontsize=12)
plt.tight_layout()
plt.show()

print("Different filters detect different features in the image!")

## Step 10: Test on Individual Examples

In [ ]:
# Get some test images
test_images, test_labels = next(iter(test_loader))

# Make predictions with both models
cnn_model.eval()
simple_model.eval()

with torch.no_grad():
    cnn_outputs = cnn_model(test_images.to(device))
    cnn_preds = cnn_outputs.argmax(dim=1).cpu()
    
    simple_outputs = simple_model(test_images.to(device))
    simple_preds = simple_outputs.argmax(dim=1).cpu()

# Find examples where they disagree
disagree_idx = (cnn_preds != simple_preds).nonzero().squeeze()

if len(disagree_idx.shape) > 0 and len(disagree_idx) > 0:
    print(f"Found {len(disagree_idx)} examples where CNN and Simple NN disagree!")
    
    # Show some disagreements
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    
    for i, ax in enumerate(axes.flat):
        if i < len(disagree_idx):
            idx = disagree_idx[i].item() if len(disagree_idx.shape) > 0 else disagree_idx.item()
            img = test_images[idx].squeeze() * 0.5 + 0.5
            true = test_labels[idx].item()
            cnn_pred = cnn_preds[idx].item()
            simple_pred = simple_preds[idx].item()
            
            ax.imshow(img, cmap='gray')
            title = f'True: {true}\nCNN: {cnn_pred}\nSimple: {simple_pred}'
            
            # Color based on which is correct
            if cnn_pred == true:
                color = 'green'
            elif simple_pred == true:
                color = 'blue'
            else:
                color = 'red'
            
            ax.set_title(title, fontsize=9)
        ax.axis('off')
    
    plt.suptitle('Cases Where CNN and Simple NN Disagree', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("In this batch, CNN and Simple NN agree on all predictions!")

## Step 11: Save the CNN Model

In [ ]:
# Save the CNN model
torch.save(cnn_model.state_dict(), 'mnist_cnn.pth')
print("CNN model saved to 'mnist_cnn.pth'")

---

## Summary

**What we learned:**

1. **Convolutions** slide filters across images to detect patterns
2. **Pooling** reduces dimensions while keeping important features
3. **CNNs** are more efficient and accurate for images
4. **Feature maps** show what the network "sees"

**Key differences: CNN vs Simple NN**

| Aspect | Simple NN | CNN |
|--------|-----------|-----|
| Input handling | Flattens image | Keeps 2D structure |
| Spatial info | Lost | Preserved |
| Parameters | More | Fewer (weight sharing) |
| Accuracy | ~97-98% | ~99%+ |
| Translation | Not invariant | Invariant |

**Key PyTorch layers for CNNs:**

| Layer | Purpose |
|-------|--------|
| `nn.Conv2d` | Apply convolution filters |
| `nn.MaxPool2d` | Reduce spatial dimensions |
| `nn.Flatten` | Convert 2D to 1D for FC layers |

---

**Congratulations!** You've now built both a simple neural network and a CNN for image classification. The CNN achieves better accuracy with fewer parameters by taking advantage of the spatial structure in images.

---